In [1]:
%cd /Users/akarsh/Documents/projects/fastInfer/StoryForge

/Users/akarsh/Documents/projects/fastInfer/StoryForge


In [2]:
import numpy as np
import torch

from pathlib import Path

from storyforge.config import load_config
from storyforge.training import load_initial_model
from storyforge.optimization import build_optimizer
from storyforge.evaluation import evaluate
from storyforge.metric import record_metrics

In [3]:
config = load_config("configs/default.toml")

In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [5]:
model = load_initial_model(config, device)
optimizer = build_optimizer(model, config)

In [6]:
train_tokens = np.load(
    config["encoding"]["train_path"],
    mmap_mode="r",
)

In [7]:
val_tokens = np.load(
    config["encoding"]["validation_path"],
    mmap_mode="r",
)

In [8]:
rng = np.random.default_rng(config["seed"])

In [9]:
print("Device:", device)
print("Loaded the initial model.")

Device: mps
Loaded the initial model.


In [10]:
metrics_path = Path(config["logging"]["metrics_path"])

In [11]:
if metrics_path.exists():
    raise FileExistsError(
        "A metrics file already exists. "
        "Use a fresh metrics path for a new training run."
    )

In [12]:
initial_validation_loss = evaluate(
    model,
    val_tokens,
    config,
    device,
)

record_metrics(
    metrics_path,
    {
        "step": 0,
        "processed_tokens": 0,
        "train_loss": None,
        "val_loss": initial_validation_loss,
        "learning_rate": None,
        "elapsed_seconds": 0.0,
    },
)

In [13]:
print("Initial validation loss:", initial_validation_loss)
print("Saved initial measurement:", metrics_path)

Initial validation loss: 9.087669491767883
Saved initial measurement: runs/baseline/metrics.jsonl


In [14]:
from storyforge.checkpoint import save_checkpoint

In [15]:
config = load_config("configs/default.toml")

In [16]:
best_val_loss = initial_validation_loss

In [17]:
save_checkpoint(
    path=config["checkpoints"]["best_path"],
    model=model,
    optimizer=optimizer,
    config=config,
    step=0,
    processed_tokens=0,
    val_loss=best_val_loss,
)

In [18]:
print("Current best validation loss:", best_val_loss)
print("Saved:", config["checkpoints"]["best_path"])

Current best validation loss: 9.087669491767883
Saved: runs/baseline/best.pt
